# Imports

In [ ]:
import behaviors
import no_signaling_sets
import numpy as np
import samplers

from tqdm import tqdm

In [62]:
delta = 2
m = 2

sampler = samplers.NoSignalingSampler(delta, m)
srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)

# Run once

In [63]:
sampled_behavior = sampler.sample()

result = srns_set.lp_test(sampled_behavior)

print(f"alpha value: {-result.fun}")

2025-05-20 11:34:22.268 | SUCCESS  | samplers:sample_multiple:100 - Samples shape: (1, 32)
2025-05-20 11:34:22.269 | DEBUG    | no_signaling_sets:express_as_function_of_q:194 - Estimated memory complexity of lacking_betas: 56 bytes


alpha value: 1.179502898600031


## Check the closest SRNS behavior

In [64]:
yielded_behavior = behaviors.LatentSRNSBehavior(
    delta=delta,
    m=m,
    vector=np.clip(np.array(result.x[1:]), 0, 1),
)

print(f"Yielded behavior is {yielded_behavior}")
print(f"Yielded behavior is tested [{yielded_behavior.is_no_signaling()}] to being no signaling")


Yielded behavior is Behavior:
Short path (z=S):
[[0.41029848 0.08236602 0.32764226 0.30151216]
 [0.02527924 0.3532117  0.05399405 0.08012414]
 [0.51152373 0.53146423 0.59417996 0.31231809]
 [0.05289854 0.03295805 0.02418373 0.30604561]]
Long path (z=L) :
[[0.27908987 0.34253306]
 [0.1379845  0.02063534]
 [0.01850335 0.0184679 ]
 [0.         0.        ]
 [0.11910556 0.05566237]
 [0.         0.11734916]
 [0.44531672 0.44535217]
 [0.         0.        ]]
------------
Yielded behavior is tested [True] to being no signaling


## Sanity check

In [65]:
print(f"PR box is tested [{srns_set.is_in_set(behaviors.pr_box)}] to being SRNS")


2025-05-20 11:34:22.285 | DEBUG    | no_signaling_sets:express_as_function_of_q:194 - Estimated memory complexity of lacking_betas: 56 bytes


PR box is tested [False] to being SRNS


# Run on a batch and color the SRNS set

In [66]:
from time import time

delta = 2
m = 2

sampler = samplers.NoSignalingSampler(delta, m)
srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)

n_samples = int(1e6)

In [ ]:
sample_list = sampler.sample_multiple(number_of_samples=n_samples)
files_suffix = f"delta_{delta}_m_{m}_samples_{n_samples}_runtime_{time()}"

2025-05-20 11:34:48.080 | SUCCESS  | samplers:sample_multiple:100 - Samples shape: (1000000, 32)


In [ ]:
np.save(f"../data/view_srns/sampled_behaviors_{files_suffix}.npy", sample_list)

In [ ]:
belonging_list = []
for i, behavior in tqdm(enumerate(sample_list)):
    belonging_list.append([i, srns_set.is_in_set(behavior)])

np.save(f"../data/view_srns/belonging_list_{files_suffix}.npy", belonging_list)

In [ ]:
analyzer = samplers.SamplesAnalyzer(
    delta=delta,
    m=m,
    sample_list=sample_list,
)

In [ ]:
analyzer.plot_projection(
    save_path=f"../data/view_srns/projection_{files_suffix}.png",
    plot=False,
    colors=belonging_list,
    )